[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Many to Many &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: the requirements and their pairs, the read-only `Student.sections`, and `audit`. Run
it first. The tasks do not depend on one another, and the last cell removes the scratch folder.


In [1]:
import logging
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, ForeignKey, MetaData, String, Table, UniqueConstraint, create_engine,
                        event, func, insert, inspect, select, text)
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, sessionmaker
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


course_requirements = Table(
    "course_requirements", Base.metadata,
    Column("course_id", ForeignKey("courses.id"), primary_key=True),
    Column("requirement_id", ForeignKey("requirements.id"), primary_key=True),
)


class Requirement(Base):
    __tablename__ = "requirements"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50), unique=True)

    courses: Mapped[list["Course"]] = relationship(secondary=course_requirements, back_populates="requirements",
                                                   order_by="Course.code")

    def __repr__(self):
        return f"Requirement({self.name!r})"


Course.requirements = relationship(Requirement, secondary=course_requirements, back_populates="courses",
                                   order_by=Requirement.name)
Base.metadata.create_all(engine)                    # creates the two new tables, and leaves the rest alone


SATISFIES = {
    "Humanities": ["HIS-110"],
    "Lab Science": ["BIO-101", "CHE-110"],
    "Quantitative Reasoning": ["CSC-101", "MAT-120", "MAT-121", "STA-200"],
    "Social Science": ["PSY-101", "STA-200"],
    "Writing": ["ENG-105", "HIS-110"],
}

with SessionLocal.begin() as session:
    courses = {course.code: course for course in session.scalars(select(Course))}
    for name, codes in SATISFIES.items():
        session.add(Requirement(name=name, courses=[courses[code] for code in codes]))


Student.sections = relationship(Section, secondary="enrollments", viewonly=True, order_by=Section.id)


def audit(session, student_id):
    """Every requirement, and the first course the student passed that meets it, or None if nothing does yet."""
    student = session.get(Student, student_id)
    passed = [enrollment.section.course for enrollment in student.enrollments
              if enrollment.status == "completed" and enrollment.grade != "F"]
    report = {}
    for requirement in session.scalars(select(Requirement).order_by(Requirement.name)):
        meeting = [course.code for course in passed if requirement in course.requirements]
        report[requirement.name] = meeting[0] if meeting else None
    return student, report


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


**1.** One requirement's courses.


In [2]:
with SessionLocal() as session:
    quantitative = session.scalars(select(Requirement).where(Requirement.name == "Quantitative Reasoning")).one()
    print(quantitative.courses)


[Course('CSC-101', 3), Course('MAT-120', 4), Course('MAT-121', 4), Course('STA-200', 3)]


**2.** A pair added from the course's side.


In [3]:
with SessionLocal.begin() as session:
    composition = session.scalars(select(Course).where(Course.code == "ENG-105")).one()
    humanities = session.scalars(select(Requirement).where(Requirement.name == "Humanities")).one()
    composition.requirements.append(humanities)

with SessionLocal() as session:
    humanities = session.scalars(select(Requirement).where(Requirement.name == "Humanities")).one()
    print(humanities.courses)


[Course('ENG-105', 3), Course('HIS-110', 3)]


Humanities now lists Composition beside World History, read in a new session, so the pair was saved
from the course's side and found from the requirement's.


**3.** Courses that meet nothing, with an outer join.


In [4]:
MEETS_NOTHING = (
    select(Course.code)
    .outerjoin(Course.requirements)
    .where(Requirement.id.is_(None))
    .order_by(Course.code)
)
with SessionLocal() as session:
    print(session.scalars(MEETS_NOTHING).all())


['CSC-201']


The outer join keeps every course, with `NULL` in the requirement's columns for a course with no
pair, and `Requirement.id.is_(None)` keeps only those. Data Structures is the one course with
nothing.


**4.** A student's sections, and then his grades.


In [5]:
with SessionLocal() as session:
    ben = session.get(Student, 2)
    print("sections:", ben.sections)
    print("finished:", [(enrollment.section, enrollment.grade) for enrollment in ben.enrollments
                        if enrollment.status == "completed"])


sections: [Section(14), Section(17), Section(20), Section(21), Section(25), Section(28), Section(32), Section(36), Section(39)]
finished: [(Section(14), 'C-'), (Section(17), 'A'), (Section(20), 'B'), (Section(21), 'D'), (Section(25), 'B+'), (Section(28), 'C+')]


**5.** A read-only relationship writes nothing.


In [6]:
BENS = select(func.count()).select_from(Enrollment).where(Enrollment.student_id == 2)

with SessionLocal() as session:
    ben = session.get(Student, 2)
    print("enrollments before:", session.scalar(BENS))
    ben.sections.append(session.get(Section, 31))
    session.flush()
    print("enrollments after: ", session.scalar(BENS), "| in the list:", len(ben.sections))
    session.rollback()


enrollments before: 9
enrollments after:  9 | in the list: 10


The list grew in Python, and the flush sent nothing, because a `viewonly` relationship never writes:
the count of Ben Okafor's enrollments did not change. The list is simply wrong until it is loaded
again.


**6.** An audit for every student who started in Fall 2025.


In [7]:
with SessionLocal() as session:
    starters = session.scalars(select(Student.id).where(Student.started_on == date(2025, 8, 25)).order_by(Student.id))
    for student_id in starters.all():
        student, report = audit(session, student_id)
        still_needed = sum(1 for code in report.values() if code is None)
        print(f"{student.name:<14} still needs {still_needed} of {len(report)}")


Chloe Martin   still needs 3 of 5
Felix Wagner   still needs 2 of 5
Isabel Costa   still needs 1 of 5
Liam Murphy    still needs 1 of 5
Olivia Brandt  still needs 2 of 5
Rosa Delgado   still needs 1 of 5
Umar Farouk    still needs 1 of 5
Yara Haddad    still needs 2 of 5


Each of them has finished one term, three courses, and a course such as World History or Statistics
meets two requirements at once, so some of them are already only one requirement short.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Many to Many](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/12-many-to-many.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
